# 01 — Catálogo y datos de evaluación

Preparamos el catálogo ESCI en español y separamos las consultas de desarrollo y evaluación. Conservamos todos los productos anotados de cada consulta; el agente recibe el catálogo sin etiquetas de relevancia.


In [ ]:
%pip -q install -r requirements-lab4-modular.txt

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import json
import pandas as pd
from lab4_asistente import config as C
from lab4_asistente.datos import preparar


## Preparar el subconjunto de trabajo

Se usan 150 consultas `train` para desarrollo y 250 consultas `test` para la evaluación final. La selección se realiza por `query_id`; se conservan todos los pares anotados de las consultas elegidas y se agregan productos distractores hasta alcanzar el mínimo solicitado.

In [ ]:
resumen_datos = preparar(
    C.RUTA_INPUT, C.RUTA_DATOS, C.RUTA_EVAL,
    n_eval=C.N_CONSULTAS_EVAL,
    n_dev=C.N_CONSULTAS_DEV,
    min_catalogo=C.MIN_CATALOGO,
    semilla=C.SEMILLA,
)
print(json.dumps(resumen_datos, indent=2, ensure_ascii=False))


## Comprobar separación y cobertura

In [ ]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
consultas = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
qrels = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")

assert len(catalogo) >= C.MIN_CATALOGO
assert (consultas["conjunto"] == "eval").sum() >= 200
assert qrels["product_id"].isin(catalogo["product_id"]).all()
assert "esci_label" not in catalogo.columns

print("Catálogo:", len(catalogo))
print("Consultas dev/eval:", consultas["conjunto"].value_counts().to_dict())
print("Pares anotados:", len(qrels))
print("Etiquetas separadas del catálogo: OK")
